In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# --- 0. CARGA DE RECURSOS (Añadido para que funcione) ---
@st.cache_data
def load_data():
    df = pd.read_csv('df_final.csv') # Asegúrate que se llame así
    df['Date'] = pd.to_datetime(df['Date'])
    return df

@st.cache_resource
def load_model():
    return joblib.load('modelo_futbol_xgboost.pkl') # Tu modelo guardado

df_matches = load_data()
modelo_final = load_model()

# --- 1. CONFIGURACIÓN DE LA PÁGINA ---
st.set_page_config(
    page_title="Predictor de Fútbol Pro", 
    page_icon="⚽",
    layout="wide"
)

# --- 2. ESTILOS PERSONALIZADOS ---
st.markdown("""
    <style>
    .stProgress > div > div > div > div { background-color: #2ecc71; }
    .main { background-color: #f8f9fa; }
    </style>
    """, unsafe_allow_html=True)

# --- 3. SIDEBAR (MODIFICADO PARA SER AUTOMÁTICO) ---
with st.sidebar:
    st.title("Menu")
    st.subheader("Selección de Partido")
    
    # Selectores de equipos reales
    lista_equipos = sorted(df_matches['HomeTeam'].unique())
    equipo_local = st.selectbox("Equipo Local", lista_equipos, index=0)
    equipo_visitante = st.selectbox("Equipo Visitante", lista_equipos, index=1)
    
    st.markdown("---")
    
    # Lógica de extracción (Tu corrección del sort_values)
    stats_h = df_matches[df_matches['HomeTeam'] == equipo_local].sort_values("Date").iloc[-1]
    stats_a = df_matches[df_matches['AwayTeam'] == equipo_visitante].sort_values("Date").iloc[-1]

    # Asignamos las variables automáticamente (Usando los nombres de tu df)
    home_elo = stats_h['Homeelo']
    away_elo = stats_a['Awayelo']
    form_home = stats_h['Form5Home']
    form_away = stats_a['Form5Away']
    odd_home = stats_h['OddHome']
    odd_away = stats_h['OddAway']
    odd_draw = stats_h['OddDraw']

    st.success(f"Datos cargados para el encuentro.")
    st.info(f"ELO Local: {int(home_elo)} | ELO Visitante: {int(away_elo)}")

# --- 4. LÓGICA DE DATOS (Conectada al modelo real) ---
# El orden de las columnas debe ser EXACTAMENTE el que usaste en X_train
data = {
    'HomeElo': [home_elo], 'AwayElo': [away_elo],
    'Form5Home': [form_home], 'Form5Away': [form_away],
    'OddHome': [odd_home], 'OddDraw': [odd_draw], 'OddAway': [odd_away],
    'Anio': [2025], 'Mes': [2], 'Dia_Semana': [5],
    'Hour_sin': [0.5], 'Hour_cos': [0.8]
}
df_input = pd.DataFrame(data)

# Realizamos la predicción real con tu modelo XGBoost
try:
    probs = modelo_final.predict_proba(df_input)[0]
except:
    probs = [0.33, 0.33, 0.34] # Fallback por si las columnas no coinciden

# --- 5. CUERPO PRINCIPAL (Tu diseño original intacto) ---
st.title("⚽ Dashboard de Predicción de Resultados")
st.caption(f"Analizando: {equipo_local} vs {equipo_visitante}")

col_main1, col_main2 = st.columns([1, 2], gap="large")

with col_main1:
    with st.container(border=True):
        st.subheader("📋 Resumen de Entrada")
        st.dataframe(df_input.T.rename(columns={0: 'Valor'}), use_container_width=True)

with col_main2:
    with st.container(border=True):
        st.subheader("🔮 Predicción del Algoritmo")
        
        clases = [f'Victoria {equipo_local} (H)', 'Empate (D)', f'Victoria {equipo_visitante} (A)']
        iconos = ['🏠', '🤝', '🚀']
        
        for i in range(len(clases)):
            col_text, col_prob = st.columns([2, 1])
            col_text.write(f"### {iconos[i]} {clases[i]}")
            col_prob.write(f"## {probs[i]*100:.1f}%")
            st.progress(probs[i])
        
        st.divider()
        ganador = clases[np.argmax(probs)]
        st.success(f"### 🎯 Resultado sugerido: **{ganador}**")

# --- 6. FOOTER / ANÁLISIS ---
st.markdown("---")
col_f1, col_f2 = st.columns(2)

with col_f1:
    st.subheader("📈 Análisis de Importancia")
    st.write(f"Comparativa de potencia entre {equipo_local} y {equipo_visitante}")
    chart_data = pd.DataFrame(
        [home_elo, away_elo],
        index=[equipo_local, equipo_visitante],
        columns=["Puntos ELO"]
    )
    st.bar_chart(chart_data)

with col_f2:
    st.subheader("📌 Nota Informativa")
    st.caption("Este modelo utiliza XGBoost para procesar el histórico de cuotas y ELO.")